In [ ]:
import pandas as pd

features_complete = pd.read_csv('../dataset_for_modeling/features_for_coach_changes.csv')
coaches = pd.read_csv('../dataset_cleaned/coaches.csv')


### Prediction Problem 2

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np


features_complete = pd.read_csv('../dataset_for_modeling/features_for_coach_changes.csv')


feature_cols_coach = [
    'win_pct_vs_league',   
    'playoff_win_pct',      
    'eff_diff',             
    'def_four_factors',     
    'career_win_pct'        
]
target_col = 'coach_changed'

LATEST_YEAR_IN_DATA = features_complete['year'].max()
VALIDATION_TEST_YEAR = 10 
OFFICIAL_PREDICT_YEAR = LATEST_YEAR_IN_DATA + 1


print("="*60)
print(f"STAGE 1: MODEL VALIDATION (TRAIN 1-{VALIDATION_TEST_YEAR-1}, TEST {VALIDATION_TEST_YEAR})")
print("="*60)

train_val = features_complete[features_complete['year'] < VALIDATION_TEST_YEAR].copy()
test_val = features_complete[features_complete['year'] == VALIDATION_TEST_YEAR].copy()

X_train_val = train_val[feature_cols_coach]
Y_train_val = train_val[target_col]
X_test_val = test_val[feature_cols_coach]
Y_test_val = test_val[target_col].copy() 

print(f"Validation Train Samples: {len(X_train_val)} (Years 1-{VALIDATION_TEST_YEAR-1})")
print(f"Validation Test Samples: {len(X_test_val)} (Year {VALIDATION_TEST_YEAR})")

model_val = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_split=10, 
    min_samples_leaf=5, max_features='sqrt', 
    class_weight='balanced', random_state=42, n_jobs=-1
)

model_val.fit(X_train_val, Y_train_val)


teams_that_changed_y11 = ['MIN', 'SAC', 'SAS', 'LAS']
y_test_val_indexed = Y_test_val.sort_index

actual_change_mask = test_val['tmID'].isin(teams_that_changed_y11)

Y_test_actual = np.where(actual_change_mask, 1, 0)

Y_pred_proba_val = model_val.predict_proba(X_test_val)[:, 1]
Y_pred_class_val = model_val.predict(X_test_val)

try:
    roc_auc_val = roc_auc_score(Y_test_actual, Y_pred_proba_val)
except ValueError:
    roc_auc_val = np.nan
    
f1_val = f1_score(Y_test_actual, Y_pred_class_val)
cm_val = confusion_matrix(Y_test_actual, Y_pred_class_val)

naive_predictions_val = np.zeros_like(Y_test_actual)
naive_f1_val = f1_score(Y_test_actual, naive_predictions_val)

print(f"\nModel Performance (Test on Year {VALIDATION_TEST_YEAR} -> Year {OFFICIAL_PREDICT_YEAR} Decision):")
print(f"  ROC AUC Score: {roc_auc_val:.4f}")
print(f"  F1 Score: {f1_val:.4f}")
print(f"  Confusion Matrix:\n{cm_val}")
print(f"\nNaive Baseline F1: {naive_f1_val:.4f}")

if f1_val > naive_f1_val:
    print(f"✓ Model F1 Score is {f1_val - naive_f1_val:.4f} better than baseline F1!")
else:
    print(f"⚠️ WARNING: Model F1 Score is NOT better than the baseline.")

validation_comparison = pd.DataFrame({
    'tmID': test_val['tmID'].values,
    'coachID': test_val['coachID'].values,
    'actual_coach_changed': Y_test_actual, 
    'predicted_class': Y_pred_class_val,
    'prob_coach_change': Y_pred_proba_val,
}).sort_values(by='prob_coach_change', ascending=False)

print("\nDetailed Validation Predictions (Year 10):")
print(validation_comparison.to_string(index=False))

print("\n" + "="*80)
print(f"STAGE 2: OFFICIAL FORECAST (TRAIN 1-{LATEST_YEAR_IN_DATA}, PREDICT {OFFICIAL_PREDICT_YEAR})")
print("="*80)

year_11_data = features_complete[features_complete['year'] == LATEST_YEAR_IN_DATA].copy()

if year_11_data.empty:
    print(f"ERROR: Cannot run official forecast. No data found for year {LATEST_YEAR_IN_DATA}.")
else:
    train_final = features_complete.copy()

    X_train_final = train_final[feature_cols_coach]
    Y_train_final = train_final[target_col]

    print(f"\nTraining Final Model on ALL {len(X_train_final)} samples...")

    model_final = RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_split=10, 
        min_samples_leaf=5, max_features='sqrt', 
        class_weight='balanced', random_state=42, n_jobs=-1
    )

    
    model_final.fit(X_train_final, Y_train_final)

    X_predict_11 = year_11_data[feature_cols_coach]

    Y_pred_proba_11 = model_final.predict_proba(X_predict_11)[:, 1]
    Y_pred_class_11 = model_final.predict(X_predict_11)

    official_forecast = pd.DataFrame({
        'tmID': year_11_data['tmID'].values,
        'coachID': year_11_data['coachID'].values,
        'stats_year': LATEST_YEAR_IN_DATA,
        'predicted_for_year': OFFICIAL_PREDICT_YEAR,
        'prob_coach_change': Y_pred_proba_11,
        'predicted_decision': Y_pred_class_11,
    }).sort_values(by='prob_coach_change', ascending=False)

    print("\nOfficial Forecast (Sorted by likelihood of change):")
    print(official_forecast.to_string(index=False))

## Check Optimal Thresholds

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
import pandas as pd
import numpy as np

thresholds = np.arange(0.50, 0.95, 0.05)
optimization_results = []

print("\n" + "="*50)
print("THRESHOLD OPTIMIZATION RESULTS")
print("="*50)

for t in thresholds:
    y_pred_tuned = (Y_pred_proba_val >= t).astype(int)
    
    try:
        f1 = f1_score(Y_test_actual, y_pred_tuned)
        precision = precision_score(Y_test_actual, y_pred_tuned)
        recall = recall_score(Y_test_actual, y_pred_tuned)
    except ValueError:
        f1, precision, recall = 0.0, 0.0, 0.0

    optimization_results.append({
        'threshold': f"{t:.2f}",
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    })
    
    print(f"Threshold {t:.2f}: F1={f1:.4f}, Precision={precision:.4f}, Recall={recall:.4f}")

opt_df = pd.DataFrame(optimization_results).sort_values('f1_score', ascending=False)
best_threshold = opt_df.iloc[0]

print("\n" + "="*50)
print(f"BEST THRESHOLD (MAX F1 Score): {best_threshold['threshold']}")
print(f"Max F1 Score: {best_threshold['f1_score']:.4f}")
print("="*50)

# Check Dataset Statistics

In [ ]:
print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)

teams_per_year = features_complete.groupby('year')['tmID'].nunique()
print("\nTeams per year:")
print(teams_per_year.to_string())

print(f"\nTotal unique teams: {features_complete['tmID'].nunique()}")
print(f"Year range: {features_complete['year'].min()} to {features_complete['year'].max()}")

# Time Series Cross-Validation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

feature_cols = [
    'win_pct_vs_league',
    'playoff_win_pct',
    'eff_diff',
    'def_four_factors',
    'career_win_pct'
]
target_col = 'coach_changed'

features_complete = features_complete.sort_values(['year', 'tmID'])

X_all = features_complete[feature_cols]
y_all = features_complete[target_col]
years_all = features_complete['year']

tscv = TimeSeriesSplit(n_splits=5)

cv_results = []

print("="*80)
print("TIME SERIES CROSS-VALIDATION (CLASSIFICATION - ROC AUC)")
print("="*80)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X_all)):
    X_train_fold = X_all.iloc[train_idx]
    y_train_fold = y_all.iloc[train_idx]
    X_test_fold = X_all.iloc[test_idx]
    y_test_fold = y_all.iloc[test_idx]
    
    test_data_with_year = features_complete.iloc[test_idx].copy()
    
    test_years = test_data_with_year['year'].unique()
    train_years = features_complete.iloc[train_idx]['year'].unique()
    
    latest_test_year = test_years.max()
    
    valid_test_mask = test_data_with_year['year'] < latest_test_year

    X_test_valid = X_test_fold[valid_test_mask]
    y_test_valid = y_test_fold[valid_test_mask]
    
    validated_years = test_data_with_year[valid_test_mask]['year'].unique()
    
    if len(y_test_valid) == 0 or len(np.unique(y_test_valid)) < 2:
        print(f"Fold {fold + 1} skipped: Not enough samples or classes for valid ROC AUC.")
        continue

    model_fold = RandomForestClassifier(
        n_estimators=200, max_depth=8, 
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    model_fold.fit(X_train_fold, y_train_fold)

    y_pred_proba_fold = model_fold.predict_proba(X_test_valid)[:, 1]
    
    roc_auc_fold = roc_auc_score(y_test_valid, y_pred_proba_fold)
    
    cv_results.append({
        'fold': fold + 1,
        'train_years': f"{train_years.min()}-{train_years.max()}",
        'test_years_evaluated': f"{validated_years.min()}-{validated_years.max()}",
        'n_train': len(y_train_fold),
        'n_test_valid': len(y_test_valid),
        'roc_auc': roc_auc_fold,
    })
    
    print(f"\nFold {fold + 1}:")
    print(f"  Train: years {train_years.min()}-{train_years.max()} ({len(y_train_fold)} samples)")
    print(f"  Test Validated: years {validated_years.min()}-{validated_years.max()} ({len(y_test_valid)} samples)")
    print(f"  ROC AUC: {roc_auc_fold:.4f}")

cv_df = pd.DataFrame(cv_results)

print("\n" + "="*80)
print("FINAL CROSS-VALIDATION SUMMARY (ROC AUC)")
print("="*80)
print("\nAll Folds:")
print(cv_df.to_string(index=False))

print(f"\n{'='*80}")
print("AVERAGE PERFORMANCE ACROSS 5 TIME SERIES FOLDS")
print(f"{'='*80}")
print(f"  Average ROC AUC: {cv_df['roc_auc'].mean():.4f} ± {cv_df['roc_auc'].std():.4f}")

# Leave One Year Out Cross-Validation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score
import pandas as pd
import numpy as np

feature_cols = [
    'win_pct_vs_league',
    'playoff_win_pct',
    'eff_diff',
    'def_four_factors',
    'career_win_pct'
]
target_col = 'coach_changed' 

lyo_results = []

available_years = sorted(features_complete['year'].unique())

testable_years = available_years[1:-1]
print("="*80)
print("EXPANDING WINDOW CROSS-VALIDATION (CLASSIFICATION - ROC AUC)")
print("="*80)
print(f"Testing features from years: {testable_years}")

for test_year in testable_years:
    train_lyo = features_complete[features_complete['year'] < test_year]
    test_lyo = features_complete[features_complete['year'] == test_year]
    
    if len(train_lyo) < 10 or len(test_lyo) == 0:
        continue
    
    X_train_lyo = train_lyo[feature_cols]
    y_train_lyo = train_lyo[target_col]
    X_test_lyo = test_lyo[feature_cols]
    y_test_lyo = test_lyo[target_col]

    if len(np.unique(y_test_lyo)) < 2:
        print(f"Year {test_year} skipped: Only one class present in the test set. (Cannot calculate ROC AUC)")
        continue
        
    model_lyo = RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_split=10, 
        min_samples_leaf=5, max_features='sqrt', 
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    model_lyo.fit(X_train_lyo, y_train_lyo)
    
    y_pred_proba_lyo = model_lyo.predict_proba(X_test_lyo)[:, 1]
    y_pred_class_lyo = model_lyo.predict(X_test_lyo)
    
    roc_auc_lyo = roc_auc_score(y_test_lyo, y_pred_proba_lyo)
    f1_lyo = f1_score(y_test_lyo, y_pred_class_lyo) 
    
    lyo_results.append({
        'features_year': test_year,
        'predicting_year': test_year + 1,
        'n_train': len(y_train_lyo),
        'n_test': len(y_test_lyo),
        'roc_auc': roc_auc_lyo,
        'f1_score': f1_lyo,    
    })
    
    print(f"\nFeatures from Year {test_year} → Predict Decision in Year {test_year + 1}:")
    print(f"  Train: {len(y_train_lyo)} samples (Years {train_lyo['year'].min()}-{train_lyo['year'].max()})")
    print(f"  Test:  {len(y_test_lyo)} samples")
    print(f"  Model ROC AUC: {roc_auc_lyo:.4f}")
    print(f"  Model F1 Score: {f1_lyo:.4f}")


if len(lyo_results) > 0:
    lyo_df = pd.DataFrame(lyo_results)
    
    print("\n" + "="*80)
    print("EXPANDING WINDOW CLASSIFICATION SUMMARY")
    print("="*80)
    print(lyo_df.to_string(index=False))
    
    print(f"\nAverage ROC AUC: {lyo_df['roc_auc'].mean():.4f} ± {lyo_df['roc_auc'].std():.4f}")
    print(f"Average F1 Score: {lyo_df['f1_score'].mean():.4f} ± {lyo_df['f1_score'].std():.4f}")

## Feature Importance Problem 2

In [ ]:
import pandas as pd
import numpy as np

print("\n" + "="*60)
print("3. FINAL MODEL FEATURE IMPORTANCE ANALYSIS")
print("="*60)

importances = pd.DataFrame({
    'feature': feature_cols_coach,
    'importance': model_final.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features (All 5 Selected):")
print(importances.to_string(index=False))

# Visualizations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

final_metrics = {
    'Metric': ['Precision', 'Recall', 'F1 Score'],
    'Value': [1.00, 0.25, 0.40] 
}
final_metrics_df = pd.DataFrame(final_metrics)

try:
    _ = importances.head()
    _ = lyo_df.head()
except NameError:
    print("Error: 'importances' or 'lyo_df' DataFrames not found in the environment. Please ensure")
    print("feature_importance.py and lyo_classification.py have been executed and generated these variables.")


plt.figure(figsize=(8, 5))
plt.bar(final_metrics_df['Metric'], final_metrics_df['Value'], color=['#3b82f6', '#f59e0b', '#10b981'])
plt.axhline(y=1.00, color='r', linestyle='--', linewidth=1, label='Perfect (1.0)')
plt.axhline(y=0.50, color='gray', linestyle=':', linewidth=1, label='Random (0.5)')
plt.ylim(0, 1.1)
plt.ylabel("Score")
plt.title("2011 Forecast Performance (Single Holdout Validation)", fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 6))

colors = {
    'def_four_factors': '#dc2626', 
    'eff_diff': '#2563eb',         
    'career_win_pct': '#f97316',   
    'win_pct_vs_league': '#10b981',
    'playoff_win_pct': '#6b7280'  
}
importances['color'] = importances['feature'].map(colors).fillna('gray')

plt.barh(importances['feature'], importances['importance'], color=importances['color'], alpha=0.8)
plt.xlabel("Importance Score (Gini)", fontsize=12)
plt.title("Feature Importance: What Drives the Firing Decision?", fontsize=14, fontweight='bold')
plt.gca().invert_yaxis() 
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax1 = plt.subplots(figsize=(12, 6))


color = '#3b82f6'
ax1.set_xlabel('Features Year → Prediction Year', fontsize=12)
ax1.set_ylabel('ROC AUC', color=color, fontsize=12)
ax1.plot(lyo_df['features_year'], lyo_df['roc_auc'], color=color, marker='o', linestyle='-', linewidth=2, label='ROC AUC')
ax1.tick_params(axis='y', labelcolor=color)
ax1.axhline(y=0.50, color='gray', linestyle='--', linewidth=1, label='Random (0.5)')
ax1.set_ylim(0.40, 1.0)


ax2 = ax1.twinx()  
color = '#f59e0b'
ax2.set_ylabel('F1 Score', color=color, fontsize=12) 
ax2.plot(lyo_df['features_year'], lyo_df['f1_score'], color=color, marker='x', linestyle='--', linewidth=2, label='F1 Score')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(0.1, 0.9)


x_labels = [f"{fy} → {py}" for fy, py in zip(lyo_df['features_year'], lyo_df['predicting_year'])]
ax1.set_xticks(lyo_df['features_year'])
ax1.set_xticklabels(x_labels, rotation=45, ha='right')


lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper right')

plt.title("Model Stability Across Time (Expanding Window CV)", fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
print("="*60)

# Compare Multiple Models

In [ ]:
from sklearn.discriminant_analysis import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
import pandas as pd
import numpy as np

try:
    from xgboost import XGBClassifier
    xgboost_available = True
except ImportError:
    xgboost_available = False
    print("⚠️  XGBoost not available, skipping")

try:
    from lightgbm import LGBMClassifier
    lightgbm_available = True
except ImportError:
    lightgbm_available = False
    print("⚠️  LightGBM not available, skipping")

print("\n" + "="*60)
print("COMPARING MULTIPLE CLASSIFICATION MODELS")
print("="*60)

models_to_compare = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        class_weight='balanced', 
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        class_weight='balanced', 
        random_state=42,
        n_jobs=-1
    ),
    'AdaBoost': AdaBoostClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ),
    'Logistic Regression (Scaled)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(solver='liblinear', penalty='l2', class_weight='balanced', random_state=42))
    ]),
    'KNN (Scaled)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1))
    ]),
    'SVC (Scaled)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=42))
    ])
}

if xgboost_available:
    pos_weight = (len(Y_train_val) - sum(Y_train_val)) / sum(Y_train_val)
    models_to_compare['XGBoost'] = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False, 
        eval_metric='logloss', 
        scale_pos_weight=pos_weight, 
        random_state=42,
        n_jobs=-1
    )

# Add LightGBM if available
if lightgbm_available:
    models_to_compare['LightGBM'] = LGBMClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        num_leaves=31,
        class_weight='balanced', 
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ) 

# Single Holdout Comparison

In [ ]:
holdout_results = []

for model_name, model_obj in models_to_compare.items():
    print(f"\nTraining {model_name}...")
    
    y_pred_proba = None 
    roc_auc_model = np.nan
    
    try:
        model_obj.fit(X_train_val, Y_train_val)
        
        if hasattr(model_obj, "predict_proba"):
             y_pred_proba = model_obj.predict_proba(X_test_val)[:, 1]
        elif model_name == 'Support Vector Machine (SVC)' and model_obj.probability:
             y_pred_proba = model_obj.predict_proba(X_test_val)[:, 1]
        else:
             print(f"  Warning: {model_name} lacks predict_proba; ROC AUC will be NaN.")
             
        y_pred_class = model_obj.predict(X_test_val)
        
        if y_pred_proba is not None:
             roc_auc_model = roc_auc_score(Y_test_actual, y_pred_proba)

        f1_model = f1_score(Y_test_actual, y_pred_class)
        accuracy_model = accuracy_score(Y_test_actual, y_pred_class)
        
        y_train_pred_class = model_obj.predict(X_train_val)
        train_f1_model = f1_score(Y_train_val, y_train_pred_class)
        train_acc_model = accuracy_score(Y_train_val, y_train_pred_class)
        
        holdout_results.append({
            'model': model_name,
            'test_roc_auc': roc_auc_model,
            'test_f1': f1_model,
            'test_accuracy': accuracy_model,
            'train_f1': train_f1_model,
            'train_accuracy': train_acc_model,
            'overfit_gap_f1': train_f1_model - f1_model
        })
        
        print(f"  Test ROC AUC: {roc_auc_model:.4f}, F1: {f1_model:.4f}, Acc: {accuracy_model:.4f}")
        print(f"  Train F1: {train_f1_model:.4f}, Acc: {train_acc_model:.4f}")
        
    except Exception as e:
        print(f"  ❌ Failed to train or evaluate: {str(e)}")

holdout_df = pd.DataFrame(holdout_results).sort_values('test_roc_auc', ascending=False)

print("\n" + "="*60)
print("SINGLE HOLDOUT CLASSIFICATION RESULTS SUMMARY")
print("="*60)
print(holdout_df.to_string(index=False))

if not holdout_df.empty:
    best_model = holdout_df.iloc[0]

    print(f"\n🏆 Best Model (by ROC AUC): {best_model['model']}")
    print(f"   ROC AUC: {best_model['test_roc_auc']:.4f}")
    print(f"   F1 Score: {best_model['test_f1']:.4f}")
else:
    print("\nNo models were successfully trained and evaluated.")

# Time Series CV Comparison

In [ ]:
from sklearn.metrics import roc_auc_score

tscv_model_results = []

for model_name, model_obj in models_to_compare.items():
    print(f"\nEvaluating {model_name} with Time Series CV...")
    
    fold_roc_aucs = []
    fold_f1_scores = []
    
    try:
        for fold, (train_idx, test_idx) in enumerate(tscv.split(X_all)):
            X_train_fold = X_all.iloc[train_idx]
            y_train_fold = y_all.iloc[train_idx]
            X_test_fold = X_all.iloc[test_idx]
            y_test_fold = y_all.iloc[test_idx]

            model_obj.fit(X_train_fold, y_train_fold)

            y_pred_proba_fold = model_obj.predict_proba(X_test_fold)[:, 1]
            y_pred_class_fold = model_obj.predict(X_test_fold) 
            
            test_data_with_year = features_complete.iloc[test_idx].copy()
            latest_test_year = test_data_with_year['year'].unique().max()
            valid_test_mask = test_data_with_year['year'] < latest_test_year

            X_test_valid = X_test_fold[valid_test_mask]
            y_test_valid = y_test_fold[valid_test_mask]

            y_pred_proba_valid = model_obj.predict_proba(X_test_valid)[:, 1]
            y_pred_class_valid = model_obj.predict(X_test_valid)
            
            if len(y_test_valid) == 0 or len(np.unique(y_test_valid)) < 2:
                continue

            # --- Evaluate ---
            roc_auc_fold = roc_auc_score(y_test_valid, y_pred_proba_valid)           
            f1_score_fold = f1_score(y_test_valid, y_pred_class_valid)
            
            fold_roc_aucs.append(roc_auc_fold)
            fold_f1_scores.append(f1_score_fold)
        
        avg_roc_auc = np.mean(fold_roc_aucs)
        std_roc_auc = np.std(fold_roc_aucs)
        avg_f1 = np.mean(fold_f1_scores) 
        std_f1 = np.std(fold_f1_scores)  
        
        avg_roc_auc_of_all_folds = cv_df['roc_auc'].mean() if 'cv_df' in locals() else 0.6606 
        
        tscv_model_results.append({
            'model': model_name,
            'avg_roc_auc': avg_roc_auc,
            'std_roc_auc': std_roc_auc,
            'avg_f1_score': avg_f1, 
            'std_f1_score': std_f1,
            'improvement_vs_baseline': avg_roc_auc - avg_roc_auc_of_all_folds
        })
        
        print(f"  Avg ROC AUC: {avg_roc_auc:.4f} ± {std_roc_auc:.4f}, Avg F1: {avg_f1:.4f} ± {std_f1:.4f}")
        
    except Exception as e:
        print(f"  ❌ Failed for {model_name}: {str(e)}")

try:
    avg_roc_auc_of_all_folds = cv_df['roc_auc'].mean()
    std_roc_auc_of_all_folds = cv_df['roc_auc'].std()
except NameError:
    # Fallback values if cv_df is not available
    avg_roc_auc_of_all_folds = 0.6606 
    std_roc_auc_of_all_folds = 0.1849 

tscv_model_results.append({
    'model': 'Naive Baseline (F1)',
    'avg_roc_auc': 0.50, 
    'std_roc_auc': 0.0,
    'avg_f1_score': 0.0000,
    'std_f1_score': 0.0000,
    'improvement_vs_baseline': 0.50 - avg_roc_auc_of_all_folds
})

tscv_model_results.append({
    'model': 'Naive Baseline (AUC Ref)',
    'avg_roc_auc': avg_roc_auc_of_all_folds,
    'std_roc_auc': std_roc_auc_of_all_folds,
    'avg_f1_score': np.nan,
    'std_f1_score': np.nan,
    'improvement_vs_baseline': 0.0
})


tscv_model_df = pd.DataFrame(tscv_model_results).sort_values('avg_roc_auc', ascending=False)

print("\n" + "="*60)
print("TIME SERIES CV RESULTS SUMMARY (ROC AUC & F1)")
print("="*60)
print(tscv_model_df.to_string(index=False))

print(f"\n🏆 Best Model (Time Series CV, by AUC): {tscv_model_df.iloc[0]['model']}")
print(f"   Average ROC AUC: {tscv_model_df.iloc[0]['avg_roc_auc']:.4f} ± {tscv_model_df.iloc[0]['std_roc_auc']:.4f}")
print(f"   Average F1 Score: {tscv_model_df.iloc[0]['avg_f1_score']:.4f} ± {tscv_model_df.iloc[0]['std_f1_score']:.4f}")

# Ensemble Methods

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score
import pandas as pd
import numpy as np

top_models = holdout_df.sort_values('test_roc_auc', ascending=False).head(5)

ensemble_predictions_proba = {}
models_for_ensemble = {}

for model_name in top_models['model']:
    model_obj = models_to_compare[model_name]
    
    try:
        model_obj.fit(X_train_val, Y_train_val)
        
        proba = model_obj.predict_proba(X_test_val)[:, 1]
        
        ensemble_predictions_proba[model_name] = proba
        models_for_ensemble[model_name] = model_obj
        
    except Exception as e:
        print(f"❌ Failed to train or get proba for {model_name}: {str(e)}")
        pass

if len(ensemble_predictions_proba) > 0:
    ensemble_avg_proba = np.mean(list(ensemble_predictions_proba.values()), axis=0)
    
    weights = {}
    for _, row in top_models.iterrows():
        if row['model'] in models_for_ensemble:
            weights[row['model']] = row['test_roc_auc']
    
    total_weight = sum(weights.values())
    weights = {k: v/total_weight for k, v in weights.items()}
    
    ensemble_weighted_proba = np.zeros(len(Y_test_actual))
    for model_name, proba in ensemble_predictions_proba.items():
        if model_name in weights:
            ensemble_weighted_proba += weights[model_name] * proba

    OPTIMAL_THRESHOLD = 0.65 

    ensemble_avg_class = (ensemble_avg_proba >= OPTIMAL_THRESHOLD).astype(int)
    ensemble_avg_f1 = f1_score(Y_test_actual, ensemble_avg_class)
    ensemble_avg_auc = roc_auc_score(Y_test_actual, ensemble_avg_proba)

    ensemble_weighted_class = (ensemble_weighted_proba >= OPTIMAL_THRESHOLD).astype(int)
    ensemble_weighted_f1 = f1_score(Y_test_actual, ensemble_weighted_class)
    ensemble_weighted_auc = roc_auc_score(Y_test_actual, ensemble_weighted_proba)
    
    print("\n" + "="*60)
    print("ENSEMBLE CLASSIFICATION RESULTS (T=0.65)")
    print("="*60)
    
    print(f"\nSimple Average Ensemble ({len(ensemble_predictions_proba)} models):")
    print(f"  Test ROC AUC: {ensemble_avg_auc:.4f}")
    print(f"  Test F1 Score: {ensemble_avg_f1:.4f}")
    
    print(f"\nWeighted Ensemble (Weighted by ROC AUC):")
    print(f"  Test ROC AUC: {ensemble_weighted_auc:.4f}")
    print(f"  Test F1 Score: {ensemble_weighted_f1:.4f}")
    
    print("\nWeights:")
    for model_name, weight in sorted(weights.items(), key=lambda x: x[1], reverse=True):
        print(f"  {model_name}: {weight:.3f}")
else:
    print("\nNo models were available to form an ensemble.")

# Additional Visualizations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

holdout_sorted = holdout_df.sort_values('test_roc_auc', ascending=False).copy()
tscv_sorted = tscv_model_df.sort_values('avg_roc_auc', ascending=False).copy()

# ----------------------------------------------------------------------
# PLOT 1: Single Holdout Comparison (Test ROC AUC)
# ----------------------------------------------------------------------
plt.figure(figsize=(12, 6))

colors_holdout = ['blue' for _ in holdout_sorted.index]
if not holdout_sorted.empty:
    winner_model = holdout_sorted.iloc[0]['model']
    
    # Identify the best performer and the baseline
    for i, model_name in enumerate(holdout_sorted['model']):
        if model_name == winner_model:
            colors_holdout[i] = '#10b981' # Green for best model
        elif 'Naive Baseline' in model_name:
            colors_holdout[i] = '#ef4444' # Red for baseline
        else:
             colors_holdout[i] = '#3b82f6' # Blue for others


plt.barh(holdout_sorted['model'], holdout_sorted['test_roc_auc'], color=colors_holdout, alpha=0.8)

plt.xlabel('ROC AUC Score (Higher is Better)', fontsize=12)
plt.title('1. Model Performance - Single Holdout Validation (ROC AUC)', fontsize=14, fontweight='bold')
plt.axvline(x=0.50, color='gray', linestyle='--', linewidth=1, label='Random Guessing')

plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


# ----------------------------------------------------------------------
# PLOT 2: Time Series CV Comparison (Average ROC AUC)
# ----------------------------------------------------------------------
plt.figure(figsize=(12, 6))

try:
    baseline_auc_ref = tscv_sorted[tscv_sorted['model'].str.contains('AUC Ref')]['avg_roc_auc'].iloc[0]
except:
    baseline_auc_ref = 0.50 # Fallback if specific baseline row is missing


colors_tscv = ['blue' for _ in tscv_sorted.index]

if not tscv_sorted.empty:
    winner_model_tscv = tscv_sorted.iloc[0]['model']
    
    for i, model_name in enumerate(tscv_sorted['model']):
        if model_name == winner_model_tscv:
            colors_tscv[i] = '#10b981' # Green for best model
        elif 'Naive Baseline' in model_name:
            colors_tscv[i] = '#ef4444' # Red for baseline
        else:
             colors_tscv[i] = '#3b82f6' # Blue for others

plt.barh(tscv_sorted['model'], tscv_sorted['avg_roc_auc'], 
         xerr=tscv_sorted['std_roc_auc'], color=colors_tscv, alpha=0.8, capsize=5)

plt.xlabel('Average ROC AUC (Higher is Better) ± Std Dev', fontsize=12)
plt.title('2. Model Performance - Time Series CV (Average ROC AUC)', fontsize=14, fontweight='bold')
plt.axvline(x=baseline_auc_ref, color='gray', linestyle='--', linewidth=2, label='Avg CV Performance', alpha=0.5)

plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


# ----------------------------------------------------------------------
# PLOT 3: Overfitting Analysis (Train F1 vs Test F1)
# ----------------------------------------------------------------------
plt.figure(figsize=(12, 6))


overfit_df = holdout_df[~holdout_df['model'].str.contains('Naive Baseline')].copy()

x_pos = np.arange(len(overfit_df))
width = 0.35

plt.bar(x_pos - width/2, overfit_df['train_f1'], width, 
        label='Train F1 Score', alpha=0.7, color='coral')
plt.bar(x_pos + width/2, overfit_df['test_f1'], width, 
        label='Test F1 Score', alpha=0.7, color='#10b981')

plt.xlabel('Model', fontsize=12)
plt.ylabel('F1 Score', fontsize=12)
plt.title('3. Overfitting Analysis: Train F1 vs Test F1 Score', fontsize=14, fontweight='bold')
plt.xticks(x_pos, overfit_df['model'], rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("VISUAL ANALYSIS COMPLETE!")
print("="*60)